In [ ]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    ConsoleSpanExporter,
    SimpleSpanProcessor,
)

from sqlite_span_exporter import SQLiteSpanExporter


provider = TracerProvider()

provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)

provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [ ]:
from opentelemetry.sdk.trace.export.in_memory_span_exporter import (
    InMemorySpanExporter,
)

memory_exporter = InMemorySpanExporter()

provider.add_span_processor(
    SimpleSpanProcessor(memory_exporter)
)

In [ ]:
from openai import OpenAI

from gitsource import GithubRepositoryDataReader
from minsearch import Index

from traced_rag import RAGTraced

from dotenv import load_dotenv
load_dotenv()

COMMIT = "8c1834d"

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = OpenAI()
traced_rag = RAGTraced(index=index, llm_client=client)

In [ ]:
memory_exporter.clear()

answer = traced_rag.rag(
    "How does the agentic loop keep calling the model until it stops?"
)

spans = memory_exporter.get_finished_spans()

print("Number of spans:", len(spans))
print("Span names:", [span.name for span in spans])

llm_span = next(span for span in spans if span.name == "llm")

print("Input tokens:", llm_span.attributes["input_tokens"])

duration_ms = (
    llm_span.end_time - llm_span.start_time
) / 1_000_000

print("LLM duration:", duration_ms, "ms")

In [ ]:
import sqlite3

conn = sqlite3.connect("traces.db")
cursor = conn.cursor()

cursor.execute("""
    SELECT name, input_tokens, output_tokens, cost
    FROM spans
""")

print("All spans:")
for row in cursor.fetchall():
    print(row)

cursor.execute("""
    SELECT
        name,
        COUNT(*) AS span_count,
        SUM(end_time - start_time) / 1000000000.0 AS total_time_seconds
    FROM spans
    WHERE name != 'rag'
    GROUP BY name
    ORDER BY total_time_seconds DESC
""")

print("\nSpan types by total time:")
for row in cursor.fetchall():
    print(row)

conn.close()